In [1]:
import hydra
from omegaconf import OmegaConf
print('hydra imported')
import os
import torch
from tqdm.auto import tqdm
from datasets.pfams import PfamDataset
print('dataset class imported')
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoTokenizer
from transformers import EsmModel

# ignore FutureWarnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from geomloss import SamplesLoss
from sklearn.neighbors import NearestNeighbors

hydra imported
dataset class imported


In [2]:
def prepare_batch(samples_list, device):
    batch = {}
    for key in samples_list[0].keys():
        values = [s[key] for s in samples_list]
        if isinstance(values[0], torch.Tensor):
            batch[key] = torch.stack(values).to(device)
        elif isinstance(values[0], (int, float)):
            batch[key] = torch.tensor(values).to(device)
        else:
            batch[key] = values
    return batch


def get_esm_embeddings(esm_model, input_ids, attention_mask):
    """Compute mean-pooled ESM embeddings."""
    with torch.no_grad():
        hidden_states = esm_model(input_ids, attention_mask=attention_mask).last_hidden_state
    mask = attention_mask.unsqueeze(-1).float()
    return (hidden_states * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


def retokenize_sequences(sequences, esm_tokenizer, max_length=64):
    """
    Decode and retokenize sequences. probably a bit silly
    """
    if sequences.dim() == 2:
        sequences = sequences.unsqueeze(1)
    
    batch_size, num_samples, seq_len = sequences.shape
    
    all_seqs = []
    for b in range(batch_size):
        for s in range(num_samples):
            seq_ids = sequences[b, s].cpu().tolist()
            seq_str = esm_tokenizer.decode(seq_ids, skip_special_tokens=False)
            all_seqs.append(seq_str)
    
    tokens = esm_tokenizer(
        all_seqs, 
        return_tensors='pt',
        padding=True, 
        truncation=True, 
        max_length=max_length
    )
    return tokens['input_ids'], tokens['attention_mask']

def sample_anytoany(encoder, generator, source_batch, target_batch, num_samples):
    with torch.no_grad():
        source_embedding = encoder(source_batch)
        target_embedding = encoder(target_batch)
        sampled_sequences = generator.sample(
            source_batch, source_embedding, target_embedding, 
            num_samples=num_samples
        )
    return sampled_sequences


def sample_onehot(encoder, generator, source_batch, target_batch, 
                  esm_model, nbrs, train_keys, num_samples, device):
    batch_size = source_batch['esm_input_ids'].size(0)
    
    with torch.no_grad():
        # Compute source ESM embeddings
        src_ids = source_batch['esm_input_ids'].view(-1, source_batch['esm_input_ids'].size(-1))
        src_mask = source_batch['esm_attention_mask'].view(-1, source_batch['esm_attention_mask'].size(-1))
        source_embs = get_esm_embeddings(esm_model, src_ids, src_mask)
        source_embs = source_embs.view(batch_size, -1, source_embs.size(-1)).mean(dim=1)
        
        # Compute target ESM embeddings
        tgt_ids = target_batch['esm_input_ids'].view(-1, target_batch['esm_input_ids'].size(-1))
        tgt_mask = target_batch['esm_attention_mask'].view(-1, target_batch['esm_attention_mask'].size(-1))
        target_embs = get_esm_embeddings(esm_model, tgt_ids, tgt_mask)
        target_embs = target_embs.view(batch_size, -1, target_embs.size(-1)).mean(dim=1)
        
        # Find nearest neighbors
        source_nn_indices = nbrs.kneighbors(source_embs.cpu().numpy())[1][:, 0]
        target_nn_indices = nbrs.kneighbors(target_embs.cpu().numpy())[1][:, 0]
        
        source_nn_keys = [train_keys[idx] for idx in source_nn_indices]
        target_nn_keys = [train_keys[idx] for idx in target_nn_indices]
        
        # Get one-hot encodings
        source_onehot = encoder({'idx': torch.tensor(source_nn_keys, device=device)})
        target_onehot = encoder({'idx': torch.tensor(target_nn_keys, device=device)})
        
        sampled_sequences = generator.sample(
            source_batch, source_onehot, target_onehot,
            num_samples=num_samples
        )
    return sampled_sequences


def compute_energy_distance(sampled_embs, target_embs):
    """Compute energy distance between two sets of embeddings."""
    loss_fn = SamplesLoss("energy", p=2)
    return loss_fn(sampled_embs, target_embs).item()


def compute_sliced_wasserstein(sampled_embs, target_embs):
    """Compute Sinkhorn (approximated Wasserstein) distance."""
    loss_fn = SamplesLoss("sinkhorn", blur=0.01, scaling=0.9)
    return loss_fn(sampled_embs, target_embs).item()


METRIC_FUNCTIONS = {
    'energy': compute_energy_distance,
    'sliced_wasserstein': compute_sliced_wasserstein,
}



def evaluate_distribution_matching(
    encoder, 
    generator, 
    dataset, 
    esm_model, 
    esm_tokenizer,
    model_type='anytoany',
    num_pairs=100,
    num_samples=16,
    metrics=('energy',),
    nbrs=None,
    train_keys=None,
    device='cuda',
    batch_size=8
):
    """
    Evaluate distribution matching quality.
    
    Args:
        encoder: Encoder model
        generator: Generator model
        dataset: Evaluation dataset
        esm_model: Pretrained ESM model for embeddings
        esm_tokenizer: ESM tokenizer
        model_type: 'anytoany' or 'onehot'
        num_pairs: Number of source-target pairs to evaluate
        num_samples: Number of samples to generate per pair
        metrics: Tuple of metric names ('energy', 'sliced_wasserstein')
        nbrs: NearestNeighbors model (required for 'onehot')
        train_keys: Sorted list of training keys (required for 'onehot')
        device: Device to use
        batch_size: Batch size for processing
    
    Returns:
        Dict with mean and SEM for each metric
    """
    # Validate inputs
    if model_type == 'onehot':
        if nbrs is None or train_keys is None:
            raise ValueError("nbrs and train_keys are required for onehot model_type")
    
    if model_type not in ('anytoany', 'onehot'):
        raise ValueError(f"Unknown model_type: {model_type}")
    
    for metric in metrics:
        if metric not in METRIC_FUNCTIONS:
            raise ValueError(f"Unknown metric: {metric}. Available: {list(METRIC_FUNCTIONS.keys())}")
    
    results = {metric: [] for metric in metrics}
    num_batches = (num_pairs + batch_size - 1) // batch_size
    
    for batch_idx in tqdm(range(num_batches), desc="Evaluating"):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, num_pairs)
        current_batch_size = end_idx - start_idx
        
        # Collect samples for batch
        source_samples_list = []
        target_samples_list = []
        for idx in range(start_idx, end_idx):
            pair = dataset[idx]
            source_samples_list.append(pair['source_samples'])
            target_samples_list.append(pair['target_samples'])
        
        source_batch = prepare_batch(source_samples_list, device)
        target_batch = prepare_batch(target_samples_list, device)
        
        # Generate samples
        if model_type == 'anytoany':
            sampled_sequences = sample_anytoany(
                encoder, generator, source_batch, target_batch, num_samples
            )
        else:  # onehot
            sampled_sequences = sample_onehot(
                encoder, generator, source_batch, target_batch,
                esm_model, nbrs, train_keys, num_samples, device
            )
        
        # Retokenize and compute embeddings for sampled sequences
        sampled_ids, sampled_mask = retokenize_sequences(sampled_sequences, esm_tokenizer)
        sampled_ids = sampled_ids.to(device)
        sampled_mask = sampled_mask.to(device)
        
        with torch.no_grad():
            sampled_embs = get_esm_embeddings(esm_model, sampled_ids, sampled_mask)
            sampled_embs = sampled_embs.view(current_batch_size, num_samples, -1)
            
            # Compute embeddings for target sequences
            target_ids = target_batch['esm_input_ids'].view(-1, target_batch['esm_input_ids'].size(-1))
            target_mask = target_batch['esm_attention_mask'].view(-1, target_batch['esm_attention_mask'].size(-1))
            target_embs = get_esm_embeddings(esm_model, target_ids, target_mask)
            target_embs = target_embs.view(current_batch_size, num_samples, -1)
        
        # Compute metrics for each pair in batch
        for i in range(current_batch_size):
            for metric in metrics:
                metric_fn = METRIC_FUNCTIONS[metric]
                results[metric].append(metric_fn(sampled_embs[i], target_embs[i]))
    
    # Aggregate results
    output = {}
    for metric in metrics:
        values = np.array(results[metric])
        output[f'{metric}_mean'] = values.mean()
        output[f'{metric}_sem'] = values.std() / np.sqrt(len(values))
    
    return output

In [3]:
# onehot pfam model

output_dir = '/orcd/data/omarabu/001/gokul/CoupledDistributionEmbeddings/'
output_dir += 'outputs/pfam_onehot_1e4_3b474421e59ef501ae348f1f3727057e'

config_path = os.path.join(output_dir, 'config.yaml')
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config not found at {config_path}")

config = OmegaConf.load(config_path)

# Detect model types
encoder_type, generator_type = ('esm', 'progen2')

best_model_path = os.path.join(output_dir, 'best_model.pt')
if not os.path.exists(best_model_path):
    raise FileNotFoundError(f"Best model not foun   d at {best_model_path}")

one_hot_encoder = hydra.utils.instantiate(config.encoder)
one_hot_generator = hydra.utils.instantiate(config.generator)

device = 'cuda'
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)

one_hot_encoder.load_state_dict(checkpoint['encoder_state_dict'])
one_hot_generator.load_state_dict(checkpoint['generator_state_dict'])

epoch = checkpoint.get('epoch', 'unknown')
loss = checkpoint.get('loss', float('nan'))

one_hot_encoder.to(device)
one_hot_generator.to(device)
one_hot_encoder.eval()
one_hot_generator.eval();

Some weights of TimeAwareEsmForFlow were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['input_cond_proj.0.bias', 'input_cond_proj.0.weight', 'input_cond_proj.2.bias', 'input_cond_proj.2.weight', 'output_cond_proj.0.bias', 'output_cond_proj.0.weight', 'output_cond_proj.2.bias', 'output_cond_proj.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# any to any pfam model

output_dir = '/orcd/data/omarabu/001/gokul/CoupledDistributionEmbeddings/'
output_dir += 'outputs/pfam_esm_dfm_6c61ec377c95ab55c2b7416301172534'

config_path = os.path.join(output_dir, 'config.yaml')
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config not found at {config_path}")

config = OmegaConf.load(config_path)

# Detect model types
encoder_type, generator_type = ('esm', 'progen2')

best_model_path = os.path.join(output_dir, 'best_model.pt')
if not os.path.exists(best_model_path):
    raise FileNotFoundError(f"Best model not found at {best_model_path}")

anytoany_encoder = hydra.utils.instantiate(config.encoder)
anytoany_generator = hydra.utils.instantiate(config.generator)

device = 'cuda'
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)

anytoany_encoder.load_state_dict(checkpoint['encoder_state_dict'])
anytoany_generator.load_state_dict(checkpoint['generator_state_dict'])

epoch = checkpoint.get('epoch', 'unknown')
loss = checkpoint.get('loss', float('nan'))

anytoany_encoder.to(device)
anytoany_generator.to(device)
anytoany_encoder.eval()
anytoany_generator.eval();

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of TimeAwareEsmForFlow were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['input_cond_proj.0.bias', 'input_cond_proj.0.weight', 'input_cond_proj.2.bias', 'input_cond_proj.2.weight', 'output_cond_proj.0.bias', 'output_cond_proj.0.weight', 'output_cond_proj.2.bias', 'output_cond_proj.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
pfam_dataset_eval = PfamDataset(data_dir='/orcd/data/omarabu/001/gokul/DistributionEmbeddings/data/pfam', 
                           data_file='eval_pfam_tokenized_data_l64_1e4_small.pt',
                           tokenize=False,
                           set_size=16,
                           max_length=64)

pfam_dataset_train = PfamDataset(data_dir='/orcd/data/omarabu/001/gokul/DistributionEmbeddings/data/pfam', 
                           data_file='pfam_tokenized_data_l64_1e4_small.pt',
                           tokenize=False,
                           set_size=16,
                           max_length=64,
                           base_dir='')

# load pretrained model
esm_model = EsmModel.from_pretrained('facebook/esm2_t6_8M_UR50D', trust_remote_code=True).cuda()

esm_tokenizer = AutoTokenizer.from_pretrained('facebook/esm2_t6_8M_UR50D', trust_remote_code=True)
esm_tokenizer.pad_token = '<pad>'
esm_tokenizer.bos_token = '<cls>'
esm_tokenizer.eos_token = '<eos>'

train_source_embeddings = {}
p = 0
while len(train_source_embeddings) < 7500:
    batch = pfam_dataset_train[p]
    if batch['source_samples']['idx'] in train_source_embeddings.keys():
        continue
    source_samples = batch['source_samples']
    for key in source_samples.keys():
        if isinstance(source_samples[key], torch.Tensor):
            source_samples[key] = source_samples[key].unsqueeze(0).to(device)
    with torch.no_grad():
        x = esm_model(source_samples['esm_input_ids'].squeeze(0),
                      attention_mask=source_samples['esm_attention_mask'].squeeze(0)).last_hidden_state
        mask = source_samples['esm_attention_mask'].unsqueeze(-1).float()
        source_embedding = (x * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        source_embedding = source_embedding.mean(axis=1)
        train_source_embeddings[batch['source_samples']['idx']] = source_embedding
    p += 1


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
train_embs_vec = torch.cat([train_source_embeddings[k] for k in sorted(train_source_embeddings.keys())], dim=0).cpu().numpy()
nbrs = NearestNeighbors(n_neighbors=1, algorithm='ball_tree').fit(train_embs_vec)

In [12]:
# for any-to-any model
n_pairs = 100

results_anytoany = evaluate_distribution_matching(
    anytoany_encoder, anytoany_generator, pfam_dataset_eval, esm_model, esm_tokenizer,
    model_type='anytoany', num_pairs=n_pairs, num_samples=16,
    metrics=['energy', 'sliced_wasserstein'], device=device
)

print(f"any-to-any energy: {results_anytoany['energy_mean']:.4f} \pm {results_anytoany['energy_sem']:.4f}")
print(f"any-to-any sliced wasserstein: {results_anytoany['sliced_wasserstein_mean']:.4f} \pm {results_anytoany['sliced_wasserstein_sem']:.4f}")

# for onehot model
results_onehot = evaluate_distribution_matching(
    one_hot_encoder, one_hot_generator, pfam_dataset_eval, esm_model, esm_tokenizer,
    model_type='onehot', num_pairs=n_pairs, num_samples=16,
    metrics=['energy', 'sliced_wasserstein'], 
    nbrs=nbrs, train_keys=sorted(list(train_source_embeddings.keys())),
    device=device
)

print(f"onehot energy: {results_onehot['energy_mean']:.4f} \pm {results_onehot['energy_sem']:.4f}")
print(f"onehot sliced wasserstein: {results_onehot['sliced_wasserstein_mean']:.4f} \pm {results_onehot['sliced_wasserstein_sem']:.4f}")

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

any-to-any energy: 0.5527 \pm 0.0450
any-to-any sliced wasserstein: 3.5115 \pm 0.1844


Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

onehot energy: 0.8603 \pm 0.0437
onehot sliced wasserstein: 4.8000 \pm 0.2026
